# Transition Validation
Thin notebook that reuses `validation_plots.shared_validation` plotting functions.

In [1]:
from pathlib import Path
import os
import subprocess
import tkinter as tk
from tkinter import filedialog
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from imu_features.config import PipelineConfig
import validation_plots.shared_validation as sv


In [2]:
DATA_ROOT = Path("Data")
PATIENT_ID = "patient_08"
DATE = None             # e.g. "2026-03-08" or None => latest
USE_TKINTER_CHOOSER = False      # set True to open folder chooser UI

PLOT_PARAMS = {
    "window_gate": {
        "window_sec": 0.60,
        "transition_min_amp_threshold": 0.10,   # fallback only
        "transition_threshold_k": 5.0,
        "transition_min_duration_s": 0.5,
    },
}

COLUMN_MAPPING_OVERRIDE = {
    "timestamp": ["Timestamp", "timestamp", "time", "Time"],
    "accel_x": ["Accel_X", "accel_x", "acc_x"],
    "accel_y": ["Accel_Y", "accel_y", "acc_y"],
    "accel_z": ["Accel_Z", "accel_z", "acc_z"],
    "useracc_x": ["UserAccel_X", "useracc_x", "linacc_x", "linearacc_x"],
    "useracc_y": ["UserAccel_Y", "useracc_y", "linacc_y", "linearacc_y"],
    "useracc_z": ["UserAccel_Z", "useracc_z", "linacc_z", "linearacc_z"],
    "gyro_x": ["Gyro_X", "gyro_x", "gyr_x"],
    "gyro_y": ["Gyro_Y", "gyro_y", "gyr_y"],
    "gyro_z": ["Gyro_Z", "gyro_z", "gyr_z"],
}

PIPELINE_CONFIG = PipelineConfig()
PIPELINE_CONFIG.column_candidates = COLUMN_MAPPING_OVERRIDE
print("Transition config ready")


Transition config ready


In [3]:
CHOSEN_DIR = None

def _activate_notebook_process_for_dialog():
    if os.name != "posix":
        return
    try:
        subprocess.run(
            [
                "osascript",
                "-e",
                (
                    'tell application "System Events" '
                    f'to set frontmost of first process whose unix id is {os.getpid()} to true'
                ),
            ],
            check=False,
            capture_output=True,
            text=True,
        )
    except Exception:
        pass

def choose_directory_tkinter(initial_dir=None):
    try:
        _activate_notebook_process_for_dialog()
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        root.lift()
        root.focus_force()
        root.update()
        root.update_idletasks()
        try:
            root.eval('tk::PlaceWindow . center')
        except Exception:
            pass
        _activate_notebook_process_for_dialog()
        selected = filedialog.askdirectory(
            parent=root,
            initialdir=str(Path(initial_dir or DATA_ROOT).resolve()),
            title="Choose activity date folder",
            mustexist=True,
        )
        root.attributes("-topmost", False)
        root.destroy()
        return Path(selected) if selected else None
    except Exception as exc:
        print(f"tkinter chooser unavailable ({exc}); using configured path.")
        return None

CHOSEN_DIR = choose_directory_tkinter(initial_dir=DATA_ROOT) if USE_TKINTER_CHOOSER else None
print("Chosen directory:", CHOSEN_DIR)


Chosen directory: None


In [4]:
sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)

resolve_date_dir = sv.resolve_date_dir
discover_activity_files = sv.discover_activity_files
load_activity_file = sv.load_activity_file
plot_walk_validation = sv.plot_walk_validation
plot_turn_validation = sv.plot_turn_validation
plot_transition_validation = sv.plot_transition_validation
plot_turn_symmetry_comparison = sv.plot_turn_symmetry_comparison
plot_turn_pair_xcorr = sv.plot_turn_pair_xcorr
plot_transition_pair_xcorr = sv.plot_transition_pair_xcorr

print("shared_validation helpers imported")


shared_validation helpers imported


## Base Directory

In [5]:
sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)

if CHOSEN_DIR is not None:
    BASE_DIR = Path(CHOSEN_DIR)
else:
    BASE_DIR = resolve_date_dir(DATA_ROOT, PATIENT_ID, DATE)
    if DATE is None:
        DATE = BASE_DIR.name

print("Base directory:", BASE_DIR)


FileNotFoundError: Patient folder not found: Data/patient_03

In [ ]:
def plot_loaded_sensor(df_loaded, sensor_name, x_col, y_col, z_col, show_x=True, show_y=True, show_z=True, show_resultant=True, source_label=None):
    if df_loaded is None:
        print(f"[SKIP] {sensor_name}: no loaded activity dataframe")
        return
    if 'time_s' not in df_loaded.columns:
        print(f"[SKIP] {sensor_name}: loaded dataframe missing time_s")
        return
    needed = [x_col, y_col, z_col]
    missing = [c for c in needed if c not in df_loaded.columns]
    if missing:
        print(f"[SKIP] {sensor_name}: missing columns {missing}")
        return

    t_plot = pd.to_numeric(df_loaded['time_s'], errors='coerce').to_numpy(dtype=float)
    x = pd.to_numeric(df_loaded[x_col], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(df_loaded[y_col], errors='coerce').to_numpy(dtype=float)
    z = pd.to_numeric(df_loaded[z_col], errors='coerce').to_numpy(dtype=float)
    resultant = np.sqrt(x**2 + y**2 + z**2)

    plt.figure(figsize=(10, 4))
    if show_x:
        plt.plot(t_plot, x, label='x', linewidth=1.0)
    if show_y:
        plt.plot(t_plot, y, label='y', linewidth=1.0)
    if show_z:
        plt.plot(t_plot, z, label='z', linewidth=1.0)
    if show_resultant:
        plt.plot(t_plot, resultant, label='resultant', linewidth=1.8, color='black')
    title_suffix = f" ({source_label})" if source_label else ''
    plt.title(f"{sensor_name}{title_suffix}")
    plt.xlabel('Time (s)')
    plt.ylabel('Value')
    plt.grid(alpha=0.3)
    plt.legend(loc='upper right', fontsize=7, framealpha=0.80, borderpad=0.25, labelspacing=0.25, handlelength=1.6)
    plt.show()


## Load Sit To Stand File

In [ ]:
sv.configure(PIPELINE_CONFIG, PLOT_PARAMS)
activity_files = discover_activity_files(BASE_DIR, ["sit_to_stand", "stand_to_sit"])
print("Discovered activity files:")
for k, v in activity_files.items():
    print(f"  {k}: {v}")
SIT_TO_STAND_PATH = activity_files.get("sit_to_stand")
SIT_TO_STAND_DF = None
SIT_TO_STAND_META = None
SIT_TO_STAND_SUMMARY = None
SIT_TO_STAND_CACHE = None
if SIT_TO_STAND_PATH is not None:
    print(f"\n[PROCESS] sit_to_stand: {SIT_TO_STAND_PATH}")
    SIT_TO_STAND_DF, SIT_TO_STAND_META = load_activity_file(SIT_TO_STAND_PATH)
    print("Loaded plot file:", SIT_TO_STAND_PATH)
else:
    print("[SKIP] sit_to_stand: not available")


## Sit To Stand Accelerometer

In [ ]:
SHOW_ACCEL_X = True
SHOW_ACCEL_Y = True
SHOW_ACCEL_Z = True
SHOW_ACCEL_RESULTANT = True
plot_loaded_sensor(SIT_TO_STAND_DF, "Accelerometer", "accel_x", "accel_y", "accel_z", SHOW_ACCEL_X, SHOW_ACCEL_Y, SHOW_ACCEL_Z, SHOW_ACCEL_RESULTANT, source_label=SIT_TO_STAND_PATH.name if SIT_TO_STAND_PATH else None)


## Sit To Stand Gyroscope

In [ ]:
SHOW_GYRO_X = True
SHOW_GYRO_Y = True
SHOW_GYRO_Z = True
SHOW_GYRO_RESULTANT = True
plot_loaded_sensor(SIT_TO_STAND_DF, "Gyroscope", "gyro_x", "gyro_y", "gyro_z", SHOW_GYRO_X, SHOW_GYRO_Y, SHOW_GYRO_Z, SHOW_GYRO_RESULTANT, source_label=SIT_TO_STAND_PATH.name if SIT_TO_STAND_PATH else None)


## Sit To Stand User Accelerometer

In [ ]:
SHOW_USERACC_X = True
SHOW_USERACC_Y = True
SHOW_USERACC_Z = True
SHOW_USERACC_RESULTANT = True
plot_loaded_sensor(SIT_TO_STAND_DF, "User Accelerometer", "useracc_x", "useracc_y", "useracc_z", SHOW_USERACC_X, SHOW_USERACC_Y, SHOW_USERACC_Z, SHOW_USERACC_RESULTANT, source_label=SIT_TO_STAND_PATH.name if SIT_TO_STAND_PATH else None)


## Sit To Stand Feature Plots

In [ ]:
if SIT_TO_STAND_DF is not None and SIT_TO_STAND_META is not None:
    SIT_TO_STAND_SUMMARY, SIT_TO_STAND_CACHE = plot_transition_validation("sit_to_stand", SIT_TO_STAND_DF, SIT_TO_STAND_META)
    print(SIT_TO_STAND_SUMMARY)
    display(pd.DataFrame([SIT_TO_STAND_SUMMARY]))
else:
    print("[SKIP] sit_to_stand feature plots: sit_to_stand file not loaded")


## Load Stand To Sit File

In [ ]:
STAND_TO_SIT_PATH = activity_files.get("stand_to_sit")
STAND_TO_SIT_DF = None
STAND_TO_SIT_META = None
STAND_TO_SIT_SUMMARY = None
STAND_TO_SIT_CACHE = None
if STAND_TO_SIT_PATH is not None:
    print(f"\n[PROCESS] stand_to_sit: {STAND_TO_SIT_PATH}")
    STAND_TO_SIT_DF, STAND_TO_SIT_META = load_activity_file(STAND_TO_SIT_PATH)
    print("Loaded plot file:", STAND_TO_SIT_PATH)
else:
    print("[SKIP] stand_to_sit: not available")


## Stand To Sit Accelerometer

In [ ]:
SHOW_ACCEL_X = True
SHOW_ACCEL_Y = True
SHOW_ACCEL_Z = True
SHOW_ACCEL_RESULTANT = True
plot_loaded_sensor(STAND_TO_SIT_DF, "Accelerometer", "accel_x", "accel_y", "accel_z", SHOW_ACCEL_X, SHOW_ACCEL_Y, SHOW_ACCEL_Z, SHOW_ACCEL_RESULTANT, source_label=STAND_TO_SIT_PATH.name if STAND_TO_SIT_PATH else None)


## Stand To Sit Gyroscope

In [ ]:
SHOW_GYRO_X = True
SHOW_GYRO_Y = True
SHOW_GYRO_Z = True
SHOW_GYRO_RESULTANT = True
plot_loaded_sensor(STAND_TO_SIT_DF, "Gyroscope", "gyro_x", "gyro_y", "gyro_z", SHOW_GYRO_X, SHOW_GYRO_Y, SHOW_GYRO_Z, SHOW_GYRO_RESULTANT, source_label=STAND_TO_SIT_PATH.name if STAND_TO_SIT_PATH else None)


## Stand To Sit User Accelerometer

In [ ]:
SHOW_USERACC_X = True
SHOW_USERACC_Y = True
SHOW_USERACC_Z = True
SHOW_USERACC_RESULTANT = True
plot_loaded_sensor(STAND_TO_SIT_DF, "User Accelerometer", "useracc_x", "useracc_y", "useracc_z", SHOW_USERACC_X, SHOW_USERACC_Y, SHOW_USERACC_Z, SHOW_USERACC_RESULTANT, source_label=STAND_TO_SIT_PATH.name if STAND_TO_SIT_PATH else None)


## Stand To Sit Feature Plots

In [ ]:
if STAND_TO_SIT_DF is not None and STAND_TO_SIT_META is not None:
    STAND_TO_SIT_SUMMARY, STAND_TO_SIT_CACHE = plot_transition_validation("stand_to_sit", STAND_TO_SIT_DF, STAND_TO_SIT_META)
    print(STAND_TO_SIT_SUMMARY)
    display(pd.DataFrame([STAND_TO_SIT_SUMMARY]))
else:
    print("[SKIP] stand_to_sit feature plots: stand_to_sit file not loaded")


## Transition Comparison

In [ ]:
transition_comparisons = []
if SIT_TO_STAND_CACHE is not None and STAND_TO_SIT_CACHE is not None:
    trans_corr = plot_transition_pair_xcorr(SIT_TO_STAND_CACHE, STAND_TO_SIT_CACHE)
    transition_comparisons.append(trans_corr)
    print(trans_corr)
else:
    print("[SKIP] transition comparison: need both sit_to_stand and stand_to_sit")
print("\n=== Validation Summary ===")
display(pd.DataFrame(([s for s in [SIT_TO_STAND_SUMMARY, STAND_TO_SIT_SUMMARY] if s is not None] + transition_comparisons)))
